# 03 — Piloto temporal (inventário de arquivos)

**Objetivo:** comparar todos os CSV baixados (linhas, colunas, soma de m³) para orientar o pipeline de **integração histórica** no anp-data-atlas.

Não consolida a série ainda — apenas diagnóstico por arquivo.

In [ ]:
from pathlib import Path

import pandas as pd

RAW_DIR = Path("../../..").resolve() / "data" / "raw" / "tancagem-abastecimento"
files = sorted(RAW_DIR.rglob("*.csv"))
print(f"{len(files)} arquivos CSV em {RAW_DIR}")

In [ ]:
rows = []
for path in files:
    try:
        df = pd.read_csv(path, encoding="utf-8")
        rows.append({
            "arquivo": str(path.relative_to(RAW_DIR)),
            "linhas": len(df),
            "colunas": len(df.columns),
            "data_distintas": df["Data"].nunique() if "Data" in df.columns else None,
            "soma_m3": df["TancagemM3"].sum() if "TancagemM3" in df.columns else None,
        })
    except Exception as e:
        rows.append({"arquivo": str(path.relative_to(RAW_DIR)), "erro": str(e)})

inv = pd.DataFrame(rows)
inv

In [ ]:
# Arquivos não-CSV (ex.: 2022 outubro xlsx)
other = [p for p in RAW_DIR.rglob("*") if p.is_file() and p.suffix.lower() not in {".csv"}]
for p in other:
    print(p.relative_to(RAW_DIR), f"({p.stat().st_size / 1024:.1f} KB)")